# 
Data Snapshot
Downloads every dataset the project uses and saves it as a local file. Every notebook onward reads these files instead of calling nflreadpy, so they all work independently from the same snapshot. Rerunning this notebook will pull the newest data. Past seasons need only be downloaded the first time. FTN charts each game within roughly 48 hours, so Wednesday/Thursday is the best day to run it to fully guaranree a full weeks data is complete.

In [1]:
import os, pandas as pd, nflreadpy as nfl, time
from datetime import datetime

CURRENT_SZN = 2026

DATA_DIR = "data/"

In [2]:
def path(name):
    return os.path.join(DATA_DIR, f"{name}.parquet")

def save(df, name):
    df.to_parquet(path(name), index = False)
    print(f"Saved {name}: {len(df):,} rows")

def get_season(loader, dataset, season, **kwargs):
    """Download one season of a dataset, or skip it if it's a past season already saved."""
    name = f"{dataset}_{season}"
    is_past = season != CURRENT_SZN
    if is_past and os.path.exists(path(name)):
        print(f"Skipped {name} (already saved)")
        return None
    for attempt in range(1, 4):
        try:
            df = loader(season, **kwargs).to_pandas()
            break
        except ConnectionError as e:
            if attempt == 3:
                raise
            print(f"{name} download failed (attempt {attempt} of 3), retrying in 10 seconds")
            time.sleep(10)
    save(df, name)
    return df

ALL_SEASONS = range(2022, (CURRENT_SZN + 1))

In [3]:
for s in ALL_SEASONS:
    get_season(nfl.load_pbp, "pbp", s)
    get_season(nfl.load_ftn_charting, "ftn", s)
    get_season(nfl.load_player_stats, "player_stats", s, summary_level="week")
    get_season(nfl.load_ff_opportunity, "ff_opportunity", s, stat_type = "weekly")

Skipped pbp_2022 (already saved)
Skipped ftn_2022 (already saved)
Skipped player_stats_2022 (already saved)
Skipped ff_opportunity_2022 (already saved)
Skipped pbp_2023 (already saved)
Skipped ftn_2023 (already saved)
Skipped player_stats_2023 (already saved)
Skipped ff_opportunity_2023 (already saved)


Saved pbp_2024: 49,492 rows
Skipped ftn_2024 (already saved)
Skipped player_stats_2024 (already saved)
Skipped ff_opportunity_2024 (already saved)


Saved pbp_2025: 48,771 rows
Skipped ftn_2025 (already saved)
Skipped player_stats_2025 (already saved)
Skipped ff_opportunity_2025 (already saved)


Saved pbp_2026: 28,467 rows


Saved ftn_2026: 27,470 rows


Saved player_stats_2026: 11,339 rows


Saved ff_opportunity_2026: 3,440 rows


## Snapshot record
Each run adds a row to `snapshot_log.csv` with the date, latest week in the data, and whether FTN has charted every game played so far. A record of what data each version of the leaderboard was built from.

In [4]:
pbp_now = pd.read_parquet(path(f"pbp_{CURRENT_SZN}"))
ftn_now = pd.read_parquet(path(f"ftn_{CURRENT_SZN}"))

reg = pbp_now[pbp_now["season_type"] == "REG"]
latest_week = int(reg["week"].max())
games_played = reg["game_id"].nunique()
games_charted = ftn_now["nflverse_game_id"].nunique()

snapshot = pd.DataFrame([{
    "downloaded_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "season": CURRENT_SZN,
    "latest_week": latest_week,
    "games_in_pbp": games_played,
    "games_in_ftn": games_charted,
}])
print(snapshot.to_string(index=False))

if games_charted < games_played:
    print(f"Note: FTN is missing {games_played - games_charted} game(s). Rerun later for full charting.")

log_path = os.path.join(DATA_DIR, "snapshot_log.csv")
snapshot.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)

   downloaded_at  season  latest_week  games_in_pbp  games_in_ftn
2026-11-24 20:41    2026           11           163           162
Note: FTN is missing 1 game(s). Rerun later for full charting.
